# MCP Chat App - Capstone Walkthrough (Weather + Airbnb)

A complete Streamlit + FastAPI + MCP application with **two MCP servers**:

- **weather** - a local server backed by the real OpenWeather API
- **airbnb**  - the community Airbnb server, run via npx (no API key)

```
Streamlit UI  ->  FastAPI backend  ->  weather + airbnb MCP servers  ->  tools
   app.py           backend.py           server.py / npx
```

This notebook builds every file and runs the MCP + agent loop live at the end.

**Running the finished app** is a two-terminal job (`uvicorn ...` and
`streamlit run ...`); those servers are launched from a terminal, not a
notebook cell. Section 8 runs the MCP core live so you can see it work here.

## Setup

In [ ]:
!pip install "mcp[cli]>=2" fastapi uvicorn streamlit openai requests python-dotenv pydantic

> Uses `mcp[cli]>=2`. The Airbnb server needs the 2.x protocol. In 2.x, `FastMCP`
> was renamed to `MCPServer`; `server.py` imports whichever is present.
>
> You also need **Node.js** installed so `npx` can run the Airbnb server.

## 1. The local weather server (`server.py`)

A tool backed by the real OpenWeather API. Its docstring becomes the tool
description; the `city` type hint becomes the input schema.

In [ ]:
%%writefile server.py
"""
Local MCP server for the capstone app.
Exposes a real weather tool backed by the OpenWeather API, over STDIO.

Run standalone for testing:  uv run server.py
Needs OPENWEATHER_API_KEY in the environment (loaded from .env).

Works on both MCP SDK versions:
  - mcp 2.x:  from mcp.server.mcpserver import MCPServer
  - mcp 1.x:  from mcp.server.fastmcp import FastMCP
This app uses 2.x because the Airbnb server requires it.
"""
import os
import requests
from dotenv import load_dotenv

load_dotenv(".env")

try:
    # MCP SDK 2.x (FastMCP was renamed to MCPServer)
    from mcp.server.mcpserver import MCPServer as _Server
except ImportError:
    # MCP SDK 1.x fallback
    from mcp.server.fastmcp import FastMCP as _Server

mcp = _Server("Weather")

OPENWEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"


@mcp.tool()
def get_weather(city: str) -> str:
    """Gets the current weather for a city using the OpenWeather API.
    city should be a plain city name, for example 'Delhi' or 'London'."""
    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        return "OPENWEATHER_API_KEY is not set on the server."
    try:
        params = {"q": city, "appid": api_key, "units": "metric"}
        resp = requests.get(OPENWEATHER_URL, params=params, timeout=10)
        if resp.status_code == 404:
            return f"City '{city}' not found."
        resp.raise_for_status()
        data = resp.json()
        desc = data["weather"][0]["description"]
        temp = data["main"]["temp"]
        feels = data["main"]["feels_like"]
        humidity = data["main"]["humidity"]
        wind = data["wind"]["speed"]
        name = data.get("name", city)
        country = data.get("sys", {}).get("country", "")
        return (
            f"Weather in {name}, {country}: {desc}. "
            f"Temperature {temp} C (feels like {feels} C), "
            f"humidity {humidity}%, wind {wind} m/s."
        )
    except Exception as e:
        return f"Error fetching weather for {city}: {e}"


if __name__ == "__main__":
    mcp.run()

## 2. Request/response models (`models.py`)

In [ ]:
%%writefile models.py
"""Pydantic models for the API's request and response shapes."""
from pydantic import BaseModel, Field


class ChatRequest(BaseModel):
    query: str = Field(..., description="The user's message")
    # The agent now sees tools from all servers and routes automatically,
    # so a server does not have to be chosen. Kept for future use.
    server: str = Field("auto", description="Reserved. Routing is automatic.")


class ChatResponse(BaseModel):
    answer: str
    tools_used: list[str] = []

## 3. The multi-server agent loop (`agent.py`)

Lists tools from every server, remembers which server owns each tool, hands the
combined list to the LLM, and routes each tool call back to the right server.

In [ ]:
%%writefile agent.py
"""
The agent loop for a multi-server setup.

Given a dict of {server_name: open ClientSession}, it:
  1. lists tools from every server and remembers which server owns each tool,
  2. hands the combined tool list to the LLM,
  3. routes each tool call to the session that owns that tool,
  4. feeds the results back and returns the final answer.

Same pattern as the course's client_query.py, extended to many servers.
"""
import json
from openai import OpenAI

# Created on first use so the app can boot before a key is needed.
_client = None


def _get_client():
    global _client
    if _client is None:
        _client = OpenAI()
    return _client


async def collect_tools(sessions: dict):
    """List tools from every session. Returns (openai_tools, owner_by_tool)."""
    openai_tools = []
    owner_by_tool = {}
    for name, session in sessions.items():
        result = await session.list_tools()
        for tool in result.tools:
            owner_by_tool[tool.name] = name
            openai_tools.append(
                {
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema,
                    },
                }
            )
    return openai_tools, owner_by_tool


async def run_agent(sessions: dict, query: str):
    """Run one query across all servers. Returns (answer, tools_used)."""
    tools_used = []
    openai_tools, owner_by_tool = await collect_tools(sessions)

    messages = [{"role": "user", "content": query}]

    response = _get_client().chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=openai_tools,
        tool_choice="auto",
    )
    messages.append(response.choices[0].message)

    if response.choices[0].message.tool_calls:
        for call in response.choices[0].message.tool_calls:
            tool_name = call.function.name
            tools_used.append(tool_name)
            # route the call to the server that owns this tool
            owner = owner_by_tool.get(tool_name)
            session = sessions[owner]
            result = await session.call_tool(
                tool_name,
                arguments=json.loads(call.function.arguments),
            )
            text = result.content[0].text if result.content else ""
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": text,
                }
            )
    else:
        return response.choices[0].message.content, tools_used

    final = _get_client().chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=openai_tools,
        tool_choice="auto",
    )
    return final.choices[0].message.content, tools_used

## 4. The FastAPI backend (`backend.py`)

Opens both servers for each request (the reliable pattern for several MCP
servers behind a web API) and exposes `/tools`, `/chat`, `/health`.

In [ ]:
%%writefile backend.py
"""
FastAPI backend for the multi-server MCP chat app.

Opens both MCP servers for the duration of each request, using the same
nested-context pattern as the course's multi-server client. This keeps the
async contexts on one task (which MCP's stdio client requires) and is the
most reliable way to use several servers behind a web API.

  - "weather" : the local server (server.py), real OpenWeather data
  - "airbnb"  : the community Airbnb server, run via npx (no API key)

Run:  uvicorn backend:app --reload
"""
import os
import sys

from dotenv import load_dotenv
from fastapi import FastAPI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from models import ChatRequest, ChatResponse
from agent import run_agent

load_dotenv(".env")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"

WEATHER = StdioServerParameters(command=sys.executable, args=["server.py"])
AIRBNB = StdioServerParameters(
    command="npx", args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"]
)

app = FastAPI(title="MCP Chat App (Weather + Airbnb)")


async def _with_sessions(fn):
    """Open both servers, run fn(sessions), then close them, all on one task."""
    async with stdio_client(WEATHER) as (r1, w1), stdio_client(AIRBNB) as (r2, w2):
        async with ClientSession(r1, w1) as weather, ClientSession(r2, w2) as airbnb:
            await weather.initialize()
            await airbnb.initialize()
            sessions = {"weather": weather, "airbnb": airbnb}
            return await fn(sessions)


@app.get("/tools")
async def list_tools():
    async def work(sessions):
        out = {}
        for name, session in sessions.items():
            result = await session.list_tools()
            out[name] = [t.name for t in result.tools]
        return out
    return await _with_sessions(work)


@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    async def work(sessions):
        answer, tools_used = await run_agent(sessions, req.query)
        return ChatResponse(answer=answer or "", tools_used=tools_used)
    return await _with_sessions(work)


@app.get("/health")
async def health():
    # Lightweight: does not open the servers.
    return {"status": "ok", "servers": ["weather", "airbnb"]}

## 5. The Streamlit UI (`app.py`)

In [ ]:
%%writefile app.py
"""
Streamlit front end. Speaks plain HTTP to the FastAPI backend, never MCP directly.

Run (after the backend is up):  streamlit run app.py
"""
import requests
import streamlit as st

API = "http://localhost:8000"

st.set_page_config(page_title="MCP Chat App", page_icon="💬")
st.title("MCP Chat App")
st.caption("Streamlit UI  ->  FastAPI  ->  Weather + Airbnb MCP servers")

if "history" not in st.session_state:
    st.session_state["history"] = []

with st.sidebar:
    st.header("Connected servers")
    try:
        tools = requests.get(f"{API}/tools", timeout=15).json()
        st.success("Backend connected")
        for server, names in tools.items():
            st.markdown(f"**{server}**")
            for t in names:
                st.markdown(f"- `{t}`")
    except Exception:
        st.error("Backend not reachable. Start it with:")
        st.code("uvicorn backend:app --reload")
    if st.button("Reset chat"):
        st.session_state["history"] = []
        st.rerun()
    st.divider()
    st.caption("Try: \"What's the weather in Delhi?\" or "
               "\"Find me an Airbnb in Goa for 2 guests.\"")

query = st.text_input("Ask something", key="query")

if st.button("Send") and query:
    with st.spinner("Thinking..."):
        try:
            r = requests.post(f"{API}/chat", json={"query": query}, timeout=180)
            data = r.json()
            st.session_state["history"].append(
                (query, data.get("answer", ""), data.get("tools_used", []))
            )
        except Exception as e:
            st.error(f"Request failed: {e}")

for user, bot, used in reversed(st.session_state["history"]):
    st.markdown(f"**You:** {user}")
    st.markdown(f"**AI:** {bot}")
    if used:
        st.caption("Tools used: " + ", ".join(used))
    st.divider()

## 6. Config and environment

Copy `.env.example` to `.env` and add your OpenAI and OpenWeather keys.

In [ ]:
%%writefile requirements.txt
mcp[cli]>=2
fastapi
uvicorn
streamlit
openai
requests
python-dotenv
pydantic

In [ ]:
%%writefile .env.example
# Copy this file to .env and paste your real keys.
OPENAI_API_KEY=sk-your-openai-key-here
OPENWEATHER_API_KEY=your-openweather-key-here

In [ ]:
%%writefile mcp.json
{
  "mcpServers": {
    "airbnb": {
      "command": "npx",
      "args": [
        "-y",
        "@openbnb/mcp-server-airbnb",
        "--ignore-robots-txt"
      ]
    }
  }
}

## 7. Running the full app (from a terminal)

Two terminals, venv active, `.env` filled in:

**Terminal 1 - backend:**
```
uvicorn backend:app --reload
```
**Terminal 2 - UI:**
```
streamlit run app.py
```
Then ask "What's the weather in Delhi?" or "Find me an Airbnb in Goa for 2 guests."

> The first Airbnb request downloads the server via npx and can take 30-60s.
> After that it is cached and quick.

## 8. Run it live in this notebook

You can watch both servers work without starting the web app. The cell below
opens both, lists their tools, and calls the weather tool.

First set your keys (or rely on a `.env` file):

In [ ]:
import os
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["OPENWEATHER_API_KEY"] = "..."
from dotenv import load_dotenv
load_dotenv(".env")
print("OpenAI key set:", bool(os.getenv("OPENAI_API_KEY")))
print("OpenWeather key set:", bool(os.getenv("OPENWEATHER_API_KEY")))

### 8a. Open both servers, list tools, call the weather tool

(The weather call needs your OpenWeather key; no OpenAI key needed for this cell.)

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

WEATHER = StdioServerParameters(command=sys.executable, args=["server.py"])
AIRBNB = StdioServerParameters(
    command="npx", args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"])

async def demo():
    async with stdio_client(WEATHER) as (r1, w1), stdio_client(AIRBNB) as (r2, w2):
        async with ClientSession(r1, w1) as weather, ClientSession(r2, w2) as airbnb:
            await weather.initialize()
            await airbnb.initialize()
            wt = await weather.list_tools()
            at = await airbnb.list_tools()
            print("weather tools:", [t.name for t in wt.tools])
            print("airbnb tools:", [t.name for t in at.tools])
            res = await weather.call_tool("get_weather", arguments={"city": "Delhi"})
            print("get_weather('Delhi') ->", res.content[0].text)

await demo()   # first run downloads the Airbnb server via npx (can take a minute)

### 8b. The full agent loop across both servers (needs your OpenAI key)

Your question goes to the model, it picks a tool from the right server, the
session runs it, and the model turns the result into a final answer.

In [ ]:
from agent import run_agent

async def ask(question):
    async with stdio_client(WEATHER) as (r1, w1), stdio_client(AIRBNB) as (r2, w2):
        async with ClientSession(r1, w1) as weather, ClientSession(r2, w2) as airbnb:
            await weather.initialize()
            await airbnb.initialize()
            sessions = {"weather": weather, "airbnb": airbnb}
            answer, used = await run_agent(sessions, question)
            print("Q:", question)
            print("Tools used:", used)
            print("A:", answer)

await ask("What's the weather in Mumbai?")
# await ask("Find me an Airbnb in Goa for 2 guests")

## Make it yours

- **Add a tool:** another `@mcp.tool()` in `server.py`, appears automatically.
- **Add a server:** add its params in `backend.py` and include it in the
  sessions dict. The agent routes by tool name.
- **Swap the model:** change `"gpt-4o"` in `agent.py`.

## Module recap

1. Foundations - MCP standardizes how AI apps reach tools and data.
2. Servers - a tool is a decorated function; its hints become the schema.
3. Clients - list and call primitives; bridge to an LLM for an agent.
4. FastAPI - turns the client into a web service.
5. Streamlit - a pure-Python UI completes the app.